In [ ]:
!pip install -q datasets tensorflow matplotlib

import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from datasets import load_dataset
from tensorflow.keras import layers as L, Model

# 1-3. Tiny ImageNet, goldfish class, text descriptions
train_data = load_dataset("zh-plus/tiny-imagenet")["train"]
label = train_data.features["label"].names.index("n01443537")
train_text, unseen_text = "a photo of a goldfish", "a close-up photo of a goldfish"

# 5-6. Load goldfish images (max 500), resize to 32x32, scale to [-1, 1]
gold = train_data.filter(lambda l: l == label, input_columns="label")
images = np.array([np.array(im.convert("RGB")) for im in gold["image"][:500]])
images = tf.image.resize(images, (32, 32)) / 127.5 - 1
print("Goldfish images loaded:", len(images))
ds = tf.data.Dataset.from_tensor_slices((images, tf.repeat(tf.constant([train_text]), len(images))))
ds = ds.shuffle(500).batch(32).prefetch(tf.data.AUTOTUNE)

# 4. Text encoder: text -> tokens -> embedding -> 64-d vector
vec = L.TextVectorization(max_tokens=100, output_sequence_length=8)
vec.adapt(tf.data.Dataset.from_tensor_slices([train_text]))
ti = L.Input((), dtype=tf.string)
enc = Model(ti, L.Dense(64, activation="relu")(
    L.GlobalAveragePooling1D()(L.Embedding(len(vec.get_vocabulary()), 32)(vec(ti)))))

# 8. Conditional generator: noise + text embedding -> 32x32 image
z, e = L.Input((100,)), L.Input((64,))
x = L.Reshape((8, 8, 128))(L.Dense(8 * 8 * 128, activation="relu")(L.Concatenate()([z, e])))
for f in (64, 32):  # 8 -> 16 -> 32
    x = L.LeakyReLU(0.2)(L.BatchNormalization()(L.Conv2D(f, 3, padding="same")(L.UpSampling2D()(x))))
G = Model([z, e], L.Conv2D(3, 3, padding="same", activation="tanh")(x))

# 9. Conditional discriminator: image + text embedding -> real/fake
im, e2 = L.Input((32, 32, 3)), L.Input((64,))
x = L.LeakyReLU(0.2)(L.Conv2D(32, 4, strides=2, padding="same")(im))
x = L.Flatten()(L.LeakyReLU(0.2)(L.BatchNormalization()(L.Conv2D(64, 4, strides=2, padding="same")(x))))
x = L.Dense(128, activation="relu")(L.Concatenate()([x, L.Dense(64, activation="relu")(e2)]))
D = Model([im, e2], L.Dense(1, activation="sigmoid")(x))

# 10-11. Loss, optimizers, adversarial training step
bce = tf.keras.losses.BinaryCrossentropy()
g_opt, d_opt = tf.keras.optimizers.Adam(2e-4, 0.5), tf.keras.optimizers.Adam(2e-4, 0.5)

@tf.function
def train_step(real, text):
    with tf.GradientTape(persistent=True) as tape:
        emb = enc(text, training=True)
        fake = G([tf.random.normal((tf.shape(real)[0], 100)), emb], training=True)
        rp, fp = D([real, emb], training=True), D([fake, emb], training=True)
        d_loss = bce(tf.ones_like(rp), rp) + bce(tf.zeros_like(fp), fp)
        g_loss = bce(tf.ones_like(fp), fp)
    gv, dv = G.trainable_variables + enc.trainable_variables, D.trainable_variables
    g_opt.apply_gradients(zip(tape.gradient(g_loss, gv), gv))
    d_opt.apply_gradients(zip(tape.gradient(d_loss, dv), dv))
    return g_loss, d_loss

# 12. Text -> image
def generate(text):
    out = G([tf.random.normal((1, 100)), enc(tf.constant([text]), training=False)], training=False)[0]
    return tf.clip_by_value((out + 1) / 2, 0, 1)

# 13. Train
EPOCHS = 15
for ep in range(EPOCHS):
    losses = np.array([[float(l) for l in train_step(r, t)] for r, t in ds])
    print(f"Epoch {ep + 1}/{EPOCHS} | G Loss: {losses[:, 0].mean():.4f} | D Loss: {losses[:, 1].mean():.4f}")
    if (ep + 1) % 5 == 0 or ep == 0:
        plt.figure(figsize=(3, 3)); plt.imshow(generate(train_text))
        plt.title(f"Goldfish - Epoch {ep + 1}"); plt.axis("off"); plt.show()

# 14-15. Unseen text description -> image
img = generate(unseen_text)
plt.figure(figsize=(5, 5)); plt.imshow(img); plt.title(unseen_text); plt.axis("off"); plt.show()
print("Text:", unseen_text, "| Generated image shape:", img.shape)

# 16. Real ImageNet example vs generated
plt.figure(figsize=(8, 4))
for i, (pic, title) in enumerate([(tf.clip_by_value((images[0] + 1) / 2, 0, 1), "Real ImageNet Goldfish"),
                                  (img, "Generated Goldfish")]):
    plt.subplot(1, 2, i + 1); plt.imshow(pic); plt.title(title); plt.axis("off")
plt.tight_layout(); plt.show()